In [2]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))
from src.models.logistic import train_logistic_model_from_csv
from src.models.gradient_boosting import gradient_boosting_pipeline
from src.models.RNN import train_rnn_from_csv
import pandas as pd


# Dataset Specifications

The dataset contains 25,721 player-game observations with 74 features, where each row represents a player’s performance in a single NBA game. The target variable is injury, a binary indicator (1 = injury, 0 = no injury).

The data is highly imbalanced, with only ~1.19% injury cases, making this a rare-event classification problem and requiring careful evaluation beyond accuracy.

### Feature Overview

Features are primarily engineered to capture player workload, fatigue, and recovery, and can be grouped into:

- Performance & Tracking Data:
minutes, distance, speed, and per-minute metrics describing in-game physical load.
- Rolling Workload Metrics:
Short- and long-term averages (e.g., 3, 5, 10, 28 games/days) capturing workload trends over time.
- Fatigue & Variability Indicators:
Percent changes, standard deviations, z-scores, and ACWR (Acute:Chronic Workload Ratio) to detect abnormal workload spikes.
- Temporal & Recovery Features:
days_since_prev, rest_days, back_to_back, short_rest capturing recovery and scheduling effects.
- Lag Features:
Previous game and recent history metrics (e.g., last 1–3 games) to model sequential injury patterns.
- Contextual Features:
home_game and matchup-related information.

### Data Quality & Preprocessing
- Most features are numerical and fully populated after preprocessing.
- Missing values are minimal and primarily limited to injury_type (expected for non-injury cases).
- Identifier and non-predictive columns (e.g., IDs, dates) are excluded during modeling.

### Summary

The dataset is designed to model injury risk as a function of workload accumulation, fatigue, and recovery, making it well-suited for both:

- Traditional models (Logistic Regression, Gradient Boosting)
- Sequential models (RNN)

In [3]:
data = pd.read_csv("../data/model_ready_data.csv")
print(data.columns)

Index(['Unnamed: 0', 'SEASON_ID', 'PLAYER_ID', 'GAME_ID', 'GAME_DATE',
       'MATCHUP', 'MIN', 'injury', 'injury_type', 'gameId', 'personId',
       'minutes', 'speed', 'distance', 'minutes_float', 'distance_per_min',
       'speed_per_min', 'rolling_distance_3', 'rolling_speed_3',
       'rolling_minutes_3', 'rolling_distance_5', 'rolling_speed_5',
       'rolling_minutes_5', 'rolling_distance_10', 'rolling_speed_10',
       'rolling_minutes_10', 'cumulative_distance', 'cumulative_minutes',
       'rolling_distance_20', 'acwr_distance', 'distance_pct_change',
       'speed_pct_change', 'distance_std_5', 'distance_z', 'speed_z',
       'minutes_prop', 'prev_game_date', 'days_since_prev', 'back_to_back',
       'distance_lag1', 'speed_lag1', 'minutes_lag1', 'distance_lag2',
       'speed_lag2', 'minutes_lag2', 'distance_lag3', 'speed_lag3',
       'minutes_lag3', 'load_intensity', 'rolling_3d_minutes_avg',
       'rolling_3d_distance_avg', 'rolling_3d_speed_avg',
       'rolling_3d_loa

# Logistic Regression Metrics

The logistic regression model was trained using SMOTE + ENN resampling to address the severe class imbalance in the dataset. A low probability threshold (0.05) was used to prioritize identifying injury cases.

Model Performance:
- ROC-AUC: 0.823
- Accuracy: 44%

Confusion Matrix:   
| 2223 | 2861 |
|------|------|
| 1    | 60   |